# Chest X-Ray Multi-Label Disease Classification
> **Deep Learning Assignment — Computer Vision: Image Classification (Current Research Areas)**  
> Model: EfficientNetV2-S | Dataset: NIH ChestX-ray14 | Framework: PyTorch

---

## Table of Contents
1. [Problem Statement](#1-problem-statement)
2. [Dataset Exploration (EDA)](#2-dataset-exploration)
3. [Model Architecture](#3-model-architecture)
4. [Training](#4-training)
5. [Results & Evaluation](#5-results--evaluation)
6. [Grad-CAM Visualization](#6-grad-cam-visualization)

## 1. Problem Statement

**Real-world problem:** Automated detection of thoracic diseases from chest X-ray images.

- **Challenge:** ~1.5 million radiologist shortage globally; manual error rate ~26%
- **Solution:** Multi-label deep learning classifier that detects 14 diseases simultaneously
- **Impact:** Reduces diagnosis time from minutes to seconds; improves early detection rates

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '../src')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from model   import ChestXrayModel
from dataset import ChestXrayDataset, CLASSES
from utils   import load_checkpoint, visualize_gradcam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 2. Dataset Exploration

In [ ]:
DATA_DIR = '../data/'
df = pd.read_csv(os.path.join(DATA_DIR, 'Data_Entry_2017.csv'))
print(f'Total images  : {len(df):,}')
print(f'Unique patients: {df["Patient ID"].nunique():,}')
df.head()

In [ ]:
# Disease distribution
label_counts = {}
for cls in CLASSES:
    label_counts[cls] = df['Finding Labels'].apply(lambda x: cls in x).sum()

label_series = pd.Series(label_counts).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(label_series.index, label_series.values, color='#0891B2', edgecolor='white')
ax.set_xlabel('Number of Images', fontsize=12)
ax.set_title('Disease Class Distribution — NIH ChestX-ray14', fontsize=14, fontweight='bold')
for bar, val in zip(bars, label_series.values):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Sample images
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
sample_imgs = df.sample(8, random_state=42)
for ax, (_, row) in zip(axes.flat, sample_imgs.iterrows()):
    img = Image.open(os.path.join(DATA_DIR, 'images', row['Image Index'])).convert('L')
    ax.imshow(img, cmap='gray')
    ax.set_title(row['Finding Labels'][:30], fontsize=7)
    ax.axis('off')
plt.suptitle('Sample X-Ray Images from NIH ChestX-ray14', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/sample_images.png', dpi=150)
plt.show()

## 3. Model Architecture

In [ ]:
model = ChestXrayModel(num_classes=14, pretrained=True)
stats = model.get_num_params()
print(f'Total parameters    : {stats["total"]:,}')
print(f'Trainable parameters: {stats["trainable"]:,}')

# Test forward pass
dummy = torch.randn(4, 3, 224, 224)
out   = model(dummy)
print(f'\nInput  shape : {dummy.shape}')
print(f'Output shape : {out.shape}   ← (batch, 14 classes)')

## 4. Training

Run from terminal:
```bash
python src/train.py --data_dir data/ --epochs 30 --batch_size 32 --lr 1e-4
```
Or execute the training loop inline below.

In [ ]:
# Quick sanity-check training (2 epochs, small batch)
from train import train_one_epoch, evaluate

train_ds = ChestXrayDataset(DATA_DIR, split='train')
val_ds   = ChestXrayDataset(DATA_DIR, split='val')
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)

model     = ChestXrayModel(num_classes=14, pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(2):
    tl = train_one_epoch(model, train_dl, criterion, optimizer, device)
    vl, auc, _ = evaluate(model, val_dl, criterion, device)
    print(f'Epoch {epoch+1}  train_loss={tl:.4f}  val_loss={vl:.4f}  AUC={auc:.4f}')

## 5. Results & Evaluation

In [ ]:
# Load per-class AUC results (generated after full training)
results = pd.read_csv('../results/per_class_auc.csv')
print(results.to_string(index=False))
print(f'\nMean AUC: {results["AUC"].mean():.4f}')

In [ ]:
# AUC bar chart
results_sorted = results.sort_values('AUC', ascending=True)
colors = ['#DC2626' if v < 0.80 else '#0891B2' if v < 0.90 else '#059669'
          for v in results_sorted['AUC']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(results_sorted['Disease'], results_sorted['AUC'], color=colors, edgecolor='white')
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5, label='AUC=0.80 threshold')
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('AUC-ROC', fontsize=12)
ax.set_title('Per-Class AUC — EfficientNetV2-S on NIH ChestX-ray14', fontsize=13, fontweight='bold')
patches = [
    mpatches.Patch(color='#059669', label='AUC ≥ 0.90'),
    mpatches.Patch(color='#0891B2', label='AUC 0.80–0.90'),
    mpatches.Patch(color='#DC2626', label='AUC < 0.80'),
]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.savefig('../results/auc_per_class.png', dpi=150)
plt.show()

In [ ]:
# Training curves
img = Image.open('../results/training_curves.png')
plt.figure(figsize=(12, 4))
plt.imshow(img); plt.axis('off')
plt.title('Training & Validation Curves', fontsize=13)
plt.show()

## 6. Grad-CAM Visualization

In [ ]:
# Load best model and generate Grad-CAM for a test image
model = ChestXrayModel(num_classes=14, pretrained=False).to(device)
load_checkpoint('../results/best_model.pth', model)

# Pick any test image
test_image = os.path.join(DATA_DIR, 'images', '00000001_000.png')
PNEUMONIA_IDX = 6

visualize_gradcam(
    model, test_image,
    class_idx=PNEUMONIA_IDX,
    class_name='Pneumonia',
    save_path='../results/gradcam/sample_gradcam.png'
)

img = Image.open('../results/gradcam/sample_gradcam.png')
plt.figure(figsize=(14, 4))
plt.imshow(img); plt.axis('off')
plt.title('Grad-CAM — Pneumonia Localization', fontsize=13)
plt.show()